# medeval Demo

PyTorch-native evaluation metrics for medical imaging: 2D/3D segmentation, classification, detection, and registration with physical spacing support, bootstrap CI, and visualization.


In [ ]:
import numpy as np
import torch
import medeval
import matplotlib

print(f"medeval: {medeval.__version__}, torch: {torch.__version__}, matplotlib: {matplotlib.__version__}, device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

## 1. Segmentation Metrics

Overlap (Dice, Jaccard, Precision, Recall, Volumetric Similarity) and surface distances (Hausdorff, HD95, ASSD) with physical spacing support.


In [ ]:
from medeval.metrics.segmentation import compute_segmentation_metrics

# 3D binary segmentation: shape (B, C, D, H, W), anisotropic spacing (dz, dy, dx) in mm
torch.manual_seed(0)
pred = (torch.rand(1, 1, 32, 64, 64) > 0.5).int()
target = (torch.rand(1, 1, 32, 64, 64) > 0.5).int()
spacing = (3.0, 1.0, 1.0)  # 3mm slice thickness, 1mm in-plane

# Basic overlap metrics
results = compute_segmentation_metrics(pred, target, spacing=spacing, include_surface=False)
print("Overlap metrics:")
for k, v in results.items():
    print(f"  {k}: {v.item():.4f}" if hasattr(v, 'item') else f"  {k}: {v:.4f}")

In [ ]:
# Surface distance metrics (use smaller volume for speed)
pred_small = torch.zeros(1, 1, 10, 20, 20)
target_small = torch.zeros(1, 1, 10, 20, 20)
pred_small[0, 0, 2:6, 5:15, 5:15] = 1  # cube
target_small[0, 0, 3:7, 6:16, 6:16] = 1  # shifted cube

results_surface = compute_segmentation_metrics(pred_small, target_small, spacing=(3.0, 1.0, 1.0), include_surface=True)
print("Surface distances (mm):")
for k, v in results_surface.items():
    unit = " mm" if "hausdorff" in k.lower() or "assd" in k.lower() else ""
    print(f"  {k}: {v.item():.4f}{unit}" if hasattr(v, 'item') else f"  {k}: {v:.4f}{unit}")

In [ ]:
# Multi-class segmentation (3 classes) with batch processing
pred_multi = torch.randint(0, 3, (4, 1, 16, 32, 32))  # 4 cases
target_multi = torch.randint(0, 3, (4, 1, 16, 32, 32))

# Reduction strategies: 'none' (per-case), 'mean-case', 'mean-class', 'global'
for reduction in ['none', 'mean-case', 'global']:
    r = compute_segmentation_metrics(pred_multi, target_multi, spacing=(2, 1, 1), reduction=reduction)
    dice = r['dice']
    if hasattr(dice, 'shape') and dice.numel() > 1:
        print(f"reduction='{reduction}': dice shape={dice.shape}")
    else:
        print(f"reduction='{reduction}': dice={dice.item():.4f}" if hasattr(dice, 'item') else f"reduction='{reduction}': dice={dice:.4f}")


## 2. Classification Metrics

AUROC, AUPRC, accuracy, F1, MCC, Cohen's kappa, calibration (ECE, Brier), with bootstrap confidence intervals.


In [ ]:
from medeval.metrics.classification import compute_classification_metrics

np.random.seed(0)
probs = np.random.rand(200)
labels = (np.random.rand(200) > 0.75).astype(int)  # ~25% positive

# Basic classification metrics
cls_results = compute_classification_metrics(probs, labels)
print("Classification metrics:")
for k, v in cls_results.items():
    if isinstance(v, tuple):  # CI tuple
        print(f"  {k}: {v[0]:.4f} [{v[1]:.4f}, {v[2]:.4f}]")
    elif hasattr(v, 'item'):
        print(f"  {k}: {v.item():.4f}")
    else:
        print(f"  {k}: {v:.4f}")

In [ ]:
# With bootstrap 95% CI and calibration metrics
cls_with_ci = compute_classification_metrics(
    probs, labels, 
    compute_ci=True, 
    confidence=0.95, 
    include_calibration=True
)
print("\nWith CI and calibration:")
for k, v in cls_with_ci.items():
    if isinstance(v, tuple) and len(v) == 3:
        print(f"  {k}: {v[0]:.4f} [95% CI: {v[1]:.4f}, {v[2]:.4f}]")
    elif hasattr(v, 'item'):
        print(f"  {k}: {v.item():.4f}")
    else:
        print(f"  {k}: {v:.4f}")

## 3. Detection Metrics

IoU, mAP, FROC for 2D/3D bounding box and instance segmentation.

In [ ]:
try:
    from medeval.metrics.detection import compute_detection_metrics
    
    # 3D bounding boxes: [x1, y1, z1, x2, y2, z2, score, class_id]
    pred_boxes = torch.tensor([
        [10, 10, 5, 30, 30, 15, 0.9, 0],
        [50, 50, 10, 70, 70, 20, 0.8, 0],
        [80, 80, 5, 100, 100, 15, 0.7, 1],
    ], dtype=torch.float32)
    
    gt_boxes = torch.tensor([
        [12, 12, 6, 32, 32, 16, 1.0, 0],
        [82, 82, 6, 102, 102, 16, 1.0, 1],
    ], dtype=torch.float32)
    
    det_results = compute_detection_metrics(pred_boxes, gt_boxes, iou_thresholds=[0.5, 0.75])
    print("Detection metrics:")
    for k, v in det_results.items():
        print(f"  {k}: {v.item():.4f}" if hasattr(v, 'item') else f"  {k}: {v:.4f}")
except ImportError:
    print("Detection module not available")


## 4. Registration Metrics

Landmark TRE, image similarity (NMI, NCC), and deformation field quality (Jacobian determinant).


In [ ]:
try:
    from medeval.metrics.registration import compute_registration_metrics
    
    # Landmark-based: predicted vs ground truth landmark positions (N, 3) in mm
    pred_landmarks = torch.tensor([[10.0, 20.0, 30.0], [40.0, 50.0, 60.0], [70.0, 80.0, 90.0]])
    gt_landmarks = torch.tensor([[10.5, 20.2, 30.1], [40.8, 50.5, 60.3], [71.0, 81.0, 91.0]])
    
    # Deformation field: (B, 3, D, H, W) displacement vectors
    deformation = torch.randn(1, 3, 16, 32, 32) * 0.1
    
    reg_results = compute_registration_metrics(
        pred_landmarks=pred_landmarks, 
        target_landmarks=gt_landmarks,
        deformation_field=deformation,
        spacing=(2.0, 1.0, 1.0)
    )
    print("Registration metrics:")
    for k, v in reg_results.items():
        if isinstance(v, dict):
            print(f"  {k}: {v}")
        elif hasattr(v, 'item'):
            unit = " mm" if "tre" in k.lower() else ""
            print(f"  {k}: {v.item():.4f}{unit}")
        elif isinstance(v, (int, float)):
            print(f"  {k}: {v:.4f}")
        else:
            print(f"  {k}: {v}")
except ImportError:
    print("Registration module not available")


## 5. Visualization

ROC/PR curves, calibration plots, and error distributions.


In [ ]:
try:
    import medeval.vis as vis
    import matplotlib.pyplot as plt
    
    # Check what's available
    available = [f for f in dir(vis) if not f.startswith('_')]
    print(f"Available vis functions: {available}")
    
    np.random.seed(42)
    y_true = np.random.randint(0, 2, 100)
    y_score = np.clip(y_true * 0.6 + np.random.randn(100) * 0.3, 0, 1)
    
    # Plot available curves
    plots = []
    if hasattr(vis, 'plot_roc_curve'):
        plots.append(('ROC', vis.plot_roc_curve))
    if hasattr(vis, 'plot_pr_curve'):
        plots.append(('PR', vis.plot_pr_curve))
    if hasattr(vis, 'plot_calibration'):
        plots.append(('Calibration', vis.plot_calibration))
    
    if plots:
        fig, axes = plt.subplots(1, len(plots), figsize=(4*len(plots), 3))
        if len(plots) == 1:
            axes = [axes]
        for ax, (name, func) in zip(axes, plots):
            func(y_true, y_score, ax=ax)
            ax.set_title(name)
        plt.tight_layout()
        plt.show()
    else:
        print("No plotting functions available")
except ImportError as e:
    print(f"Visualization module not available: {e}")


## 6. Aggregation & Stratification

Bootstrap CI and stratified aggregation by site/scanner.


In [ ]:
try:
    from medeval.core.aggregate import bootstrap_ci, stratified_aggregate
    
    # Per-case metrics from 20 samples across 2 sites
    np.random.seed(0)
    dice_scores = np.random.rand(20) * 0.3 + 0.6  # Dice scores 0.6-0.9
    site_ids = np.array([0] * 10 + [1] * 10)  # Use integer IDs
    
    # Bootstrap 95% CI
    mean, ci_low, ci_high = bootstrap_ci(dice_scores, confidence=0.95, n_bootstrap=1000)
    print(f"Dice: {mean:.4f} [95% CI: {ci_low:.4f}, {ci_high:.4f}]")
    
    # Stratified by site (metrics must be a dict)
    metrics_dict = {'dice': dice_scores}
    strat_results = stratified_aggregate(metrics_dict, site_ids)
    print(f"\nStratified by site:")
    for stratum, metrics in strat_results.items():
        print(f"  Site {stratum}: {metrics}")
except ImportError:
    print("Aggregate module not available")


## 7. CLI

Batch evaluation from command line with YAML/JSON config.


In [ ]:
# CLI usage example (run in terminal)
print("""CLI examples:
  medeval segmentation --pred pred.nii.gz --target target.nii.gz --spacing 1,1,3
  medeval classification --pred probs.csv --target labels.csv --ci
  medeval detection --pred boxes.json --target gt.json --iou 0.5
  medeval --config eval_config.yaml  # Batch evaluation from config file
""")


## 8. IO & Interoperability

Supports NumPy, PyTorch, NIfTI (SimpleITK), and optional MONAI/torchmetrics interop.


In [ ]:
try:
    from medeval.core.io import load_image, get_spacing_from_header
    from medeval.core.typing import to_tensor
    
    # Convert between formats
    arr_np = np.random.rand(32, 64, 64)
    arr_torch = to_tensor(arr_np)  # NumPy -> PyTorch
    print(f"NumPy {arr_np.shape} -> PyTorch {arr_torch.shape}")
    
    # GPU support (if available)
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    arr_gpu = to_tensor(arr_np, device=device)
    print(f"Device: {arr_gpu.device}")
except ImportError as e:
    print(f"IO module: {e}")
